# Block Order LMMs (RQ4)

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.patches import PathPatch, Patch
from matplotlib.lines import Line2D

import seaborn as sns

import statsmodels.formula.api as smf
from scipy.stats import chi2

In [ ]:
CONDS = {"Nelo": "Baseline",
        "Coda": "Proactive",
        "Lumo": "Optional"}


CONDITIONS   = ["Baseline", "Proactive", "Optional"]
DVS          = ["trust", "understanding", "nasa"]
DV_LABELS    = {"trust": "Trust", "understanding": "System Understanding", "nasa": "Mental Workload"}
BLOCKS       = [1, 2, 3]
ALPHA_BONF   = 0.05 / 3
SUBJECT_COL  = "participant_id"

COLORS = {
    "Optional": "#6929c4",
    "Proactive": "#1192e8",
    "Baseline": "#005d5d",
}

sns.set_style("whitegrid")

plt.rcParams['font.family'] = 'sans-serif'


## Load and merge data

In [ ]:
blocks = pd.read_pickle("../DATA/blocks.pkl")
demo = pd.read_csv("../DATA/demo.csv")

blocks = blocks[blocks["block"] != "trial"]
df = blocks.merge(demo, on="participant_id")

df["route"] = df["route"].astype("category")

df['condition'] = df['condition'].replace({
    'Nelo': 'Baseline',
    'Coda': 'Proactive',
    'Lumo': 'Optional'
})

df["condition"] = df["condition"].astype("category")

## Block LMMs - Numerical Block Number

In [ ]:
dependent_vars = [
    "trust",
    "understanding",
    "nasa"
]

df["run"] = df["block"].astype(int)

def lr_test(model_small, model_large):
    lr = 2 * (model_large.llf - model_small.llf)
    df_diff = model_large.df_modelwc - model_small.df_modelwc
    p = chi2.sf(lr, df_diff)
    return lr, df_diff, p


for dv in dependent_vars:

    print("=" * 80)
    print(f"DEPENDENT VARIABLE: {dv}")
    print("=" * 80)

    # small
    m0 = smf.mixedlm(
        f"{dv} ~ condition",
        data=df,
        groups=df["participant_id"]
    ).fit(reml=False)

    # main
    m1 = smf.mixedlm(
        f"{dv} ~ condition + run",
        data=df,
        groups=df["participant_id"]
    ).fit(reml=False)

    # interaction
    m2 = smf.mixedlm(
        f"{dv} ~ condition * run",
        data=df,
        groups=df["participant_id"]
    ).fit(reml=False)

    # likelihood ratio tests
    lr1, df1, p1 = lr_test(m0, m1)
    lr2, df2, p2 = lr_test(m1, m2)

    print("\n")
    print("----- Model comparison -----")
    print(f"Condition vs Condition + Block")
    print(f"LR = {lr1:.3f}, df = {df1}, p = {p1:.3f}")

    print(f"\nCondition + Block vs Interaction")
    print(f"LR = {lr2:.3f}, df = {df2}, p = {p2:.3f}")

    print("\n")
    print("----- Primary model -----")
    print(m1.summary())

    print("\n")
    print("----- Interaction model -----")
    print(m2.summary())

    print("\n\n")

## Block LMMs - Categorical Block Number

In [ ]:
df["run_cat"] = df["run"].astype("category")

for dv in dependent_vars:

    print("=" * 80)
    print(f"DEPENDENT VARIABLE: {dv}")
    print("=" * 80)

    # small
    m0 = smf.mixedlm(
        f"{dv} ~ condition",
        data=df,
        groups=df["participant_id"]
    ).fit(reml=False)

    # main
    m1 = smf.mixedlm(
        f"{dv} ~ condition + run_cat",
        data=df,
        groups=df["participant_id"]
    ).fit(reml=False)

    # interaction
    m2 = smf.mixedlm(
        f"{dv} ~ condition * run_cat",
        data=df,
        groups=df["participant_id"]
    ).fit(reml=False)


    # likelihood ratio tests
    lr1, df1, p1 = lr_test(m0, m1)
    lr2, df2, p2 = lr_test(m1, m2)

    print("\n")
    print("----- Model comparison -----")
    print(f"Condition vs Condition + Block")
    print(f"LR = {lr1:.3f}, df = {df1}, p = {p1:.3f}")

    print(f"\nCondition + Block vs Interaction")
    print(f"LR = {lr2:.3f}, df = {df2}, p = {p2:.3f}")

    print("\n")
    print("----- Primary model -----")
    print(m1.summary())

    print("\n")
    print("----- Interaction model -----")
    print(m2.summary())

    print("\n\n")

## Boxplot Visualisation

In [ ]:
means = df.groupby(["run", "condition"])["trust"].mean().reset_index()
overall_mean = df.groupby("run")["trust"].mean().reset_index()

legend_handles = []
legend_labels = []

for condition, color in COLORS.items():
    cond_data = means[means["condition"] == condition]
    if cond_data.empty:
        continue

sns.set_style("whitegrid")

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Roboto']




overall_mean = (
    df.groupby("run", as_index=False)["trust"]
      .mean()
)

fig = plt.figure(figsize=(6, 3.5), dpi=300)
ax = sns.boxplot(
        data=df,
        x='run',
        y='trust',
        hue="condition",
        hue_order=CONDITIONS,
        palette=COLORS,
        width=0.6,
        fliersize=0,
        linewidth=1,
        boxprops=dict(alpha=0.45),
        medianprops=dict(color="#111", linewidth=1),
        whiskerprops=dict(color="#666", linewidth=1),
        capprops=dict(color="#666", linewidth=1)
    )


boxes = [p for p in ax.patches if isinstance(p, PathPatch)]
for i, box in enumerate(boxes):
    condition = CONDITIONS[i // 3]
    colour = COLORS[condition]
    box.set_edgecolor(colour)
    box.set_linewidth(1.2)

legend_handles = []
for condition in CONDITIONS:
    color = COLORS[condition]

    handle = Patch(
        facecolor=to_rgba(color, alpha=0.45),
        edgecolor=color,
        linewidth=1.2
    )
    legend_handles.append(handle)

ax.grid(axis="y", color="#ddd", linewidth=0.5, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#ccc")
ax.spines["bottom"].set_color("#ccc")

plt.xlabel("Block Number", fontsize=12)
plt.ylabel("STS-AD Score", fontsize=12)
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax = plt.gca()
ax.tick_params(axis='x', which='both', length=4, width=1, color='black')
ax.tick_params(axis='y', which='both', length=4, width=1, color='black', direction='out', labelsize=10)

ax.legend().remove()


mean_color = "#9f1853"

line_m, = ax.plot(
    overall_mean["run"] - 1,
    overall_mean["trust"],
    color=mean_color,
    linestyle=":",
    marker="D",
    markersize=4,
    markerfacecolor="white",
    markeredgecolor=mean_color,
    linewidth=1.5,
    zorder=5
)

top_line_m, = ax.plot(
    overall_mean["run"] - 1,
    overall_mean["trust"],
    linestyle="None",
    marker="D",
    markersize=4,
    markerfacecolor=to_rgba(mean_color, alpha=0.4),
    markeredgecolor="None",
    zorder=6
)

legend_handles.append((line_m, top_line_m))
legend_labels.append("Mean")




legend_handles.append(
    Line2D(
        [0], [0],
        color=mean_color,
        linestyle="--",
        marker="D",
        markerfacecolor="white",
        markeredgecolor=mean_color,
        linewidth=1.5,
        label="Mean"
    )
)

leg = fig.legend(
    handles=legend_handles,
    labels=CONDITIONS + ["Mean"],
    bbox_to_anchor=(0.5, -0.05, 0., 0.02),
    loc='upper center',
    ncols=4,
    labelcolor="#222222",
    fontsize=10,
    frameon=False
)

plt.show()
